<a href="https://colab.research.google.com/github/larryjay007/MyML/blob/main/w04_baseline_score.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-07 — Baseline Action Score and Top-20 Review

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

In [ ]:
%pip -q install duckdb huggingface_hub

In [ ]:
import os, getpass
HF_TOKEN = os.environ.get('HF_TOKEN') or getpass.getpass('Paste your Hugging Face READ token (hf_...): ')

Paste your Hugging Face READ token (hf_...): ··········


In [ ]:
import duckdb

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = 'hf://datasets/FlyRank/internship-warehouse'
MARCH = f"read_parquet('{REL}/fact_content_daily_performance/month=2026-03/*.parquet')"

In [ ]:
features = con.sql(f"""
    SELECT
        content_hash_id,
        client_hash_id,
        SUM(gsc_impressions) AS impressions_mar,
        SUM(gsc_clicks) AS clicks_mar,
        AVG(NULLIF(gsc_avg_position, 0)) AS avg_position_mar,
        SUM(gsc_clicks) * 1.0 / NULLIF(SUM(gsc_impressions), 0) AS ctr_mar,
        COUNT(DISTINCT CASE WHEN gsc_impressions > 0 THEN report_date END) AS active_days_mar
    FROM {MARCH}
    GROUP BY content_hash_id, client_hash_id
    HAVING impressions_mar >= 10
""").df()

print(f"{len(features):,} pages with at least minimal March activity")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

143,206 pages with at least minimal March activity


In [ ]:
import pandas as pd

features['position_tier'] = pd.cut(
    features['avg_position_mar'],
    bins=[0, 3, 10, 20, 50, float('inf')],
    labels=['1-3', '4-10', '11-20', '21-50', '50+']
)

signal1 = features.groupby('position_tier', observed=True).agg(
    n=('ctr_mar', 'count'),
    mean_ctr=('ctr_mar', 'mean')
).reset_index()

signal1

,position_tier,n,mean_ctr
0,1-3,9109,0.004145
1,4-10,63198,0.003922
2,11-20,29345,0.002552
3,21-50,31165,0.001619
4,50+,10366,0.000618


Signal 1 — CTR vs. position tier: CONFIRMED.

Mean CTR decreases monotonically from 0.41% (position 1-3, n=9,109) down to 0.06%
(position 50+, n=10,366), with all five tiers well-populated. This confirms the premise
behind FlyRank's CTR-fix flag: position and CTR are genuinely related in this slice, so
comparing a page's CTR against its own tier's average is a legitimate way to spot
underperformance, not just borrowed logic.

In [ ]:
features['volume_tier'] = pd.cut(
    features['impressions_mar'],
    bins=[0, 50, 200, 1000, 5000, float('inf')],
    labels=['10-50', '50-200', '200-1000', '1000-5000', '5000+']
)

signal2 = features.groupby('volume_tier', observed=True).agg(
    n=('active_days_mar', 'count'),
    mean_active_days=('active_days_mar', 'mean')
).reset_index()

signal2

,volume_tier,n,mean_active_days
0,10-50,27483,12.011098
1,50-200,31014,23.632940
2,200-1000,39674,28.223648
3,1000-5000,31745,29.548559
4,5000+,13290,30.441836


Signal 2 — impression volume vs. active days: CONFIRMED.

Mean active days rises steadily from 12.0 (10-50 impressions, n=27,483) to 30.4
(5000+ impressions, n=13,290) — essentially daily presence at the highest tier. This
confirms impressions_mar reflects sustained visibility, not a single lucky spike day,
which is what the volume-behind-quick-win logic assumes.

The rule: flag a page for title/meta review if it has real, observed visibility
(impressions_mar) and its measured CTR falls meaningfully below what pages at its own
position tier typically get. This is a directional, decision-support signal — a page
underperforming its tier's expected CTR is a plausible review candidate, not proof the
title is the cause. A reviewer still makes the final call.

Reason code: ctr_underperformance_visible
Action label: review_title_meta

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write `work/outputs/baseline_action_score.csv`.*

In [ ]:
tier_expected_ctr = signal1.set_index('position_tier')['mean_ctr']

features['expected_ctr'] = features['position_tier'].map(tier_expected_ctr).astype(float)
features['ctr_gap'] = (features['expected_ctr'] - features['ctr_mar']).clip(lower=0)

features['score'] = features['impressions_mar'] * features['ctr_gap']
features['reason_code'] = 'ctr_underperformance_visible'
features['action'] = 'review_title_meta'

queue = features.sort_values('score', ascending=False).reset_index(drop=True)

import os
os.makedirs('work/outputs', exist_ok=True)
queue.to_csv('work/outputs/baseline_action_score.csv', index=False)

print(f"Queue written: {len(queue):,} rows")
queue[['content_hash_id', 'impressions_mar', 'avg_position_mar', 'ctr_mar', 'expected_ctr', 'score', 'reason_code', 'action']].head(10)

Queue written: 143,206 rows


,content_hash_id,impressions_mar,avg_position_mar,ctr_mar,expected_ctr,score,reason_code,action
0,content_44f34c0a90047651,212404.0,7.346909,0.000113,0.003922,809.114048,ctr_underperformance_visible,review_title_meta
1,content_8d7d99f109e19aa2,203497.0,2.563756,0.001420,0.004145,554.405699,ctr_underperformance_visible,review_title_meta
2,content_8e1334d6356668e3,134984.0,4.545582,0.000007,0.003922,528.448912,ctr_underperformance_visible,review_title_meta
3,content_34a70fea29d15f24,143019.0,3.219473,0.000301,0.003922,517.964662,ctr_underperformance_visible,review_title_meta
4,content_fec55986a1868d62,124075.0,9.385150,0.000008,0.003922,485.660447,ctr_underperformance_visible,review_title_meta
5,content_7c6373141eae744a,132593.0,5.789019,0.000626,0.003922,437.070672,ctr_underperformance_visible,review_title_meta
6,content_acbcc847f8996314,170808.0,3.361195,0.001534,0.003922,407.961697,ctr_underperformance_visible,review_title_meta
7,content_f6116743b00afc2d,107584.0,9.536301,0.000139,0.003922,406.977654,ctr_underperformance_visible,review_title_meta
8,content_b99ea6861864dea5,194337.0,4.450106,0.001858,0.003922,401.249697,ctr_underperformance_visible,review_title_meta
9,content_f43118e089ecc69a,139417.0,5.036458,0.001370,0.003922,355.836506,ctr_underperformance_visible,review_title_meta


In [ ]:
top20 = queue.head(20)[['content_hash_id', 'impressions_mar', 'avg_position_mar', 'ctr_mar', 'expected_ctr', 'score']]
top20

,content_hash_id,impressions_mar,avg_position_mar,ctr_mar,expected_ctr,score
0,content_44f34c0a90047651,212404.0,7.346909,0.000113,0.003922,809.114048
1,content_8d7d99f109e19aa2,203497.0,2.563756,0.001420,0.004145,554.405699
2,content_8e1334d6356668e3,134984.0,4.545582,0.000007,0.003922,528.448912
3,content_34a70fea29d15f24,143019.0,3.219473,0.000301,0.003922,517.964662
4,content_fec55986a1868d62,124075.0,9.385150,0.000008,0.003922,485.660447
5,content_7c6373141eae744a,132593.0,5.789019,0.000626,0.003922,437.070672
6,content_acbcc847f8996314,170808.0,3.361195,0.001534,0.003922,407.961697
7,content_f6116743b00afc2d,107584.0,9.536301,0.000139,0.003922,406.977654
8,content_b99ea6861864dea5,194337.0,4.450106,0.001858,0.003922,401.249697
9,content_f43118e089ecc69a,139417.0,5.036458,0.001370,0.003922,355.836506


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

| # | Page | Action | Reason code | Confidence note | What would make this wrong |
|---|---|---|---|---|---|
| 1 | 44f34c0a | review_title_meta | ctr_underperformance_visible | High — 212K impressions, position 7.3, CTR 0.011% vs 0.39% expected. Largest gap in the queue. | If this page is a reference/resource type where users scan without clicking (e.g. a definition box), low CTR may be expected behavior, not a title problem. |
| 2 | 8d7d99f1 | review_title_meta | ctr_underperformance_visible | High — position 2.6 (near-top) but CTR only 0.14% vs 0.41% expected. | If a competing result (featured snippet, ad, or another page from the same site) is absorbing clicks at this exact query, the title isn't the cause. |
| 3 | 8e1334d6 | review_title_meta | ctr_underperformance_visible | Very high — CTR is 0.0007%, essentially zero, at a strong position (4.5). | Could indicate a tracking/measurement issue (e.g. broken click tracking) rather than a real title problem — worth a manual check before assuming it's content-related. |
| 4 | 34a70fea | review_title_meta | ctr_underperformance_visible | High — CTR 0.03% vs 0.39% expected, position 3.2. | Same tracking-anomaly caveat as #3 — a CTR this close to zero at this volume is unusual enough to double-check the raw click data first. |
| 5 | fec55986 | review_title_meta | ctr_underperformance_visible | High — CTR near-zero (0.0008%) at position 9.4. | Position 9.4 is at the edge of page 1 — if this query has unusually strong competing results (ads, shopping panel), low CTR may reflect the SERP, not the title. |
| 6 | 7c637314 | review_title_meta | ctr_underperformance_visible | Moderate-high — CTR 0.06% vs 0.39% expected. | If word count or content type differs sharply from others in this tier, the tier's "expected" CTR may not be a fair comparison for this specific page type. |
| 7 | acbcc847 | review_title_meta | ctr_underperformance_visible | Moderate — CTR 0.15% vs 0.39%, position 3.4 (strong). | Smaller relative gap than #1–5; could just be normal variance within the tier rather than a real problem. |
| 8 | f6116743 | review_title_meta | ctr_underperformance_visible | High — CTR near-zero (0.014%) at position 9.5. | Same SERP-competition caveat as #5 — worth checking what else ranks for this page's main query before assuming the title is at fault. |
| 9 | b99ea686 | review_title_meta | ctr_underperformance_visible | Moderate — CTR 0.19% vs 0.39%, position 4.5. | Gap is real but smaller than the top cases — lower urgency, worth batching with similar pages rather than treating as a standalone fix. |
| 10 | f43118e0 | review_title_meta | ctr_underperformance_visible | Moderate — CTR 0.14% vs 0.39%, position 5.0. | Same as #9 — moderate gap, not an extreme outlier. |
| 11 | cd3d932d | review_title_meta | ctr_underperformance_visible | High — CTR near-zero (0.0045%) at position 7.8. | Lower impressions than #1–10 (89K) — still worth reviewing, but the absolute traffic being lost is smaller, so priority should trail the higher-volume cases above it. |
| 12 | 046fc480 | review_title_meta | ctr_underperformance_visible | High — CTR near-zero (0.0072%) at position 7.3. | Same lower-volume caveat as #11. |
| 13 | 9540d884 | review_title_meta | ctr_underperformance_visible | High — CTR 0.013% at position 7.8. | Same tracking-anomaly caveat as #3/#4 — a CTR this close to zero deserves a raw-data sanity check first. |
| 14 | 306bc78d | review_title_meta | ctr_underperformance_visible | High — position 1.5 (top-tier), but CTR only 0.043% vs 0.41% expected — the highest bar in the queue. | A position this strong with near-zero CTR is unusual — worth checking whether this is a non-clickable result type (e.g. a knowledge panel source) rather than a normal listing. |
| 15 | e578ac84 | review_title_meta | ctr_underperformance_visible | Moderate — CTR 0.14% vs 0.39%, position 4.1. | Moderate gap; could be normal variance rather than a real problem. |
| 16 | 9ef3d751 | review_title_meta | ctr_underperformance_visible | High — position 2.5 (top-tier), CTR 0.10% vs 0.41% expected. | Same top-tier caveat as #14 — check for a non-standard result type before assuming it's the title. |
| 17 | 425715 | review_title_meta | ctr_underperformance_visible | High — CTR near-zero (0.0042%) at position 6.4, but lower impressions (71K) than the top cases. | Smallest-volume page in the top 20 — real gap, but lowest absolute traffic loss of the group. |
| 18 | 36fc1ee5 | review_title_meta | ctr_underperformance_visible | Moderate-high — CTR 0.022% at position 6.5. | Similar tracking-anomaly caveat — worth a raw click-data check given how close to zero this is. |
| 19 | 4977e90c | review_title_meta | ctr_underperformance_visible | Moderate — CTR 0.038% at position 7.4. | Moderate confidence — gap is real but not the most extreme in the list. |
| 20 | 77276ad7 | review_title_meta | ctr_underperformance_visible | Moderate — CTR 0.17% vs 0.39%, position 3.9. | Smallest relative gap in the top 20 — arguably borderline for inclusion; worth a second look at whether K=20 is cutting off too generously here. |

Recurring caveat across #3, #4, #13, #18: CTR values this close to zero (under 0.05%) at
real volume are unusual enough that a raw click-tracking check should happen before
assuming the title/meta is the cause — this could be a measurement artifact rather than a
genuine performance problem.

Recurring caveat across #1, #2, #14, #16: pages already at strong or top-tier positions
(1-3) with major CTR gaps may be affected by SERP features (featured snippets, knowledge
panels) that reduce clicks regardless of title quality — position and CTR interact with
what else is on the results page, which this rule can't see.

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

Weakest picks in the top 20:

#20 (77276ad7) — smallest relative CTR gap in the top 20 (0.17% vs 0.39% expected). This
sits closer to normal tier variance than a real problem, and is arguably a borderline
inclusion — worth checking whether K=20 pulled in a page that doesn't really belong.

#3, #4, #13, #18 — CTR values this close to zero are unusual enough that they may reflect
a measurement/tracking issue rather than a genuine title problem. These need a raw-data
check before a reviewer trusts the "review_title_meta" action at face value.

Leakage check: confirmed no leakage. Every input to the score (impressions_mar, ctr_mar,
avg_position_mar, position_tier, expected_ctr) is built entirely from March data. No
product decision flags (health_score, priority_score, action_type) were used. No
April/future-window data was used anywhere in the scoring logic — is_declining was not
referenced at all in this notebook.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.